In [19]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve

In [20]:
# 1. Load Data (Replace with your actual paths if needed)
train_df = pd.read_csv(r"C:\Tata AI Hackathon\Tata-AI-Hackathon\dataset\train.csv")
test_df = pd.read_csv(r"C:\Tata AI Hackathon\Tata-AI-Hackathon\dataset\test.csv")

In [21]:
X_train = train_df.drop(columns=['CoilID', 'Y'])
y_train = train_df['Y']
X_test = test_df.drop(columns=['CoilID'])
test_ids = test_df['CoilID']

In [22]:
# Fill NaNs with a unique value so CatBoost recognizes them as missing/special
X_train = X_train.fillna(-999)
X_test = X_test.fillna(-999)

In [23]:
# 2. Setup CatBoost
# scale_pos_weight is the ratio of negative to positive samples (approx 1286/66 = 19.5)
# We set it slightly higher to enforce strict recall
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=4,
    scale_pos_weight=22, 
    l2_leaf_reg=5,
    random_state=42,
    verbose=0
)

In [24]:
# 3. Out-Of-Fold Cross-Validation
print("Running Stratified K-Fold CV with CatBoost...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    model.fit(X_tr, y_tr)
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

Running Stratified K-Fold CV with CatBoost...


In [25]:
# 4. Find the Optimal Threshold 
precisions, recalls, thresholds = precision_recall_curve(y_train, oof_preds)

# Find thresholds where recall is exactly 1.0
valid_mask = recalls[:-1] >= 1.0

if valid_mask.any():
    valid_thresholds = thresholds[valid_mask]
    valid_precisions = precisions[:-1][valid_mask]
    
    # Of the thresholds that give 1.0 recall, find the one that gives the HIGHEST precision
    best_idx = np.argmax(valid_precisions)
    best_threshold = valid_thresholds[best_idx]
    best_precision = valid_precisions[best_idx]
    print(f"Found threshold for 100% Recall. Precision at this threshold: {best_precision:.4f}")
else:
    print("Warning: Could not achieve 100% recall on OOF data.")
    best_threshold = thresholds[0]

# Small safety margin
final_threshold = max(0.0001, best_threshold - 0.005)
print(f"Applying threshold: {final_threshold:.6f}")

Found threshold for 100% Recall. Precision at this threshold: 0.0676
Applying threshold: 0.000100


In [26]:
# 5. Retrain on Full Data
print("Retraining on full dataset...")
model.fit(X_train, y_train)
test_probs = model.predict_proba(X_test)[:, 1]

Retraining on full dataset...


In [27]:
# 6. Apply and Save
test_preds = (test_probs >= final_threshold).astype(int)

submission = pd.DataFrame({
    'CoilID': test_ids,
    'Y': test_preds
})

submission.to_csv('expected_submission_v4.csv', index=False)
print("✅ Submission saved as 'expected_submission_v4.csv'")
print(f"Total defects predicted in test set: {test_preds.sum()} out of {len(test_preds)}")

✅ Submission saved as 'expected_submission_v4.csv'
Total defects predicted in test set: 339 out of 339
